Analysis:
1. What challenges did you encounter when provisioning and executing machine learning workflows in a managed cloud environment (such as Colab Enterprise or Compute Engine) compared to local or standard free-tier notebooks?

  I found that for this project, executing machine learning workflows in colab enterprise did not change much as opposed to running them in standard local colab, other than the ability to throttle access to people also involoved in developing the pipeline who are apart of the project, and the credit system in enterprise, which is more present than in standard local colab.

2. How did you identify and handle implicit missing values (such as -200 sentinel indicators) across the sensor columns, and how did your imputation strategy affect model stability?

  I handled the -200 sentinel indicators by replacing them with null values, then replacing all null values in the numeric columns with the median of said columns. I did not need a different strategy for the categorical columns because there were no missing or off values. My imputation stragety did not drastically affect stability, with the train RMSE of .4956 being still higher than the test RMSE of .4675.

3. Explain how the Scikit-Learn ColumnTransformer and Pipeline streamline the data preparation process and prevent data leakage across preprocessing and training steps in your cloud pipeline.

  Without ColumnTransformer, I would have had to write separate scaling and encoding code for the numeric columns and for Date and Time, then stitch the results together by hand. With it, I put a StandardScaler pipeline on the numeric columns and a OneHotEncoder pipeline on the categorical ones, and one fit_transform call handles both. It prevents data leakage because I fit it on X_train only. The scaler’s mean and standard deviation come from the training data and are just reused on X_test instead of being recalculated.

4. How did comparing StandardScaler versus MinMaxScaler impact your model training convergence, gradient descent optimization, and final evaluation metrics?
  I only used StandardScaler, so I didn’t directly compare it to MinMaxScaler. Scaling was needed anyway, since the PT08 sensor columns are in the hundreds or thousands while AH is around 1, and that would have made gradient descent converge poorly. I think StandardScaler was the better fit because the sensor columns have outliers, and MinMaxScaler would have merged most readings into a small range since it scales off the min and max. The SGD results also showed how sensitive convergence is to the learning rate even with scaled data. With log_loss, eta0=0.001 took 655 epochs while eta0=0.1 took 26, and both reached 0.8603 accuracy.

5. How does configuring SGDClassifier differ from standard closed-form solvers, and what advantages do probability outputs and evaluation metrics provide when assessing classification performance?
  SGDClassifier updates the weights one sample at a time, so I had to pick the loss, the learning rate (eta0) and the stopping rules myself. The softmax LogisticRegression needed none of that and reached the same accuracy in 73 iterations. It isn’t truly closed-form either, since it uses the lbfgs solver, and the only closed-form model here is LinearRegression. The log_loss and modified_huber losses give probabilities through predict_proba, while hinge only outputs a class label. Probabilities show how confident each prediction is, so I could rank readings by risk or adjust the threshold instead of only getting a label. Accuracy and macro F1 were almost the same at around 0.86, which makes sense because the CO terciles are balanced.
6. Based on your regularized regression models (Ridge versus Lasso), how did adjusting the regularization hyperparameter affect feature selection and coefficient shrinkage, and what operational environmental insights do these findings reveal?
  Ridge shrank every coefficient but never set any to zero, and test RMSE stayed around 0.4675 until alpha reached 10, then climbed to 0.6446 at alpha=10000. Lasso removed features completely. At alpha=0.05 it kept five features, at 0.3 it kept PT08.S1(CO), C6H6 and NOx, and at alpha=1 only C6H6 was left, with an RMSE of 1.1794. Benzene and NOx stayed in the model the longest, which suggests that CO moves together with other combustion and traffic pollutants, so a small set of sensors could capture most of the signal. Temperature, humidity and ozone were dropped first. These coefficients show correlation, not cause, and benzene and NOx are readings from the reference analyzers rather than low-cost sensors.